# fwi_forecast 02 — Evaluation
Criteria set in `FICHE.md` before running: median MAE at +1, +2, +3 ≤ persistence for both A and B; 10–90 % coverage in [0.70, 0.90]; 100-member forecast < 100 ms.
Test seasons are never used for fitting the climate or the boosting models. Confidence intervals: block bootstrap over weeks.

In [1]:
import sys, pathlib, time
import numpy as np
ROOT = pathlib.Path.cwd().resolve().parents[1]
sys.path[:0] = [str(ROOT / "python/src"), str(ROOT.parent / "common/python/src"), str(ROOT.parents[1] / "box/fwi/python/src"), str(ROOT.parent / "risk_gru/python/src")]
from fwi_forecast import Climate, FwiForecaster, experiments as E

## 1. Synthetic multi-year (evidence level S)
60 simulated seasons: 40 to fit, 20 to test (about 3 600 test origins).

In [2]:
ss = E.seasons(60, seed=11)
clim = Climate.fit(ss[:40])
tr = E.stack(ss[:40], clim)
te = E.stack(ss[40:], clim, with_A=True, members=100)
models = E.fit_B(tr["X"], tr["Y"])
B = E.predict_B(models, te["X"])
Y, bl = te["Y"], te["blocks"]
def show(name, pred):
    r = E.mae_row(pred, Y, bl)
    print(f"{name:14s}", "   ".join(f"+{h+1}d MAE {m:5.2f} [{lo:5.2f},{hi:5.2f}]" for h, (m, lo, hi) in enumerate(r)))
show("persistence", te["persistence"]); show("repeat today", te["repeat"]); show("A (hybrid)", te["A_q"][:, 1, :]); show("B (boosting)", B[:, 1, :])
print("10-90% coverage  A:", np.round(E.coverage(te["A_q"], Y), 3), " B:", np.round(E.coverage(B, Y), 3))

persistence    +1d MAE  6.84 [ 6.47, 7.22]   +2d MAE  8.93 [ 8.41, 9.50]   +3d MAE  9.97 [ 9.41,10.57]
repeat today   +1d MAE  6.89 [ 6.51, 7.26]   +2d MAE  9.13 [ 8.60, 9.73]   +3d MAE 10.32 [ 9.72,10.92]
A (hybrid)     +1d MAE  5.97 [ 5.66, 6.28]   +2d MAE  7.35 [ 7.00, 7.75]   +3d MAE  7.82 [ 7.41, 8.24]


B (boosting)   +1d MAE  5.91 [ 5.58, 6.22]   +2d MAE  7.21 [ 6.83, 7.60]   +3d MAE  7.74 [ 7.36, 8.14]
10-90% coverage  A: [0.791 0.77  0.765]  B: [0.775 0.783 0.782]


**Reading.** Both A and B beat persistence by a clear margin at every lead. A and B are statistically tied, which is the point of the design: A reaches B's accuracy while keeping the physics exact and needing no boosting library on the gateway.
**Caveat.** The synthetic generator and method A share the same *kinds* of structure (seasonality, AR anomalies, a Markov rain chain); the test is easier than reality.

## 2. Probability of a high-danger day
`p_high` is the share of members with FWI ≥ 21.3 (class "high"). We check it with a Brier score against persistence-as-probability.

In [3]:
from gw_common import brier, average_precision, auc
yhigh = (Y >= 21.3).astype(float)
for h in range(3):
    print(f"+{h+1}d  Brier A {brier(yhigh[:, h], te['A_phigh'][:, h]):.3f} | AUROC A {auc(yhigh[:, h], te['A_phigh'][:, h]):.3f} | AUROC persistence {auc(yhigh[:, h], te['persistence'][:, h]):.3f} | base rate {yhigh[:, h].mean():.2f}")

+1d  Brier A 0.130 | AUROC A 0.894 | AUROC persistence 0.874 | base rate 0.46
+2d  Brier A 0.164 | AUROC A 0.835 | AUROC persistence 0.803 | base rate 0.46
+3d  Brier A 0.178 | AUROC A 0.806 | AUROC persistence 0.760 | base rate 0.46


## 3. Real weather check — Algerian, leave-one-site-out (evidence level A)
The climate is a *single mean per variable* from the other site (122 days cannot define a seasonal cycle). About 116 origins per site: report only, no pass level.
Two variants, because section 1 of notebook 01 showed that **our engine does not reproduce the published codes of this data set**:
- **(i) published codes**: state and truth are the published values (another implementation) while the ensemble goes through our engine → the method is judged against a different physics.
- **(ii) own engine**: state and truth are computed by our engine from the real weather → this isolates the quality of the *weather forecast* on real weather.

In [4]:
alg = ROOT.parents[1] / "data/algerian/Algerian_forest_fires_dataset_UPDATE.csv"
for label, own in (("(i) published codes", False), ("(ii) own engine on real weather", True)):
    print("\n#####", label)
    ra = E.algerian_check(alg, own_engine=own)
    for site, v in ra.items():
        print(" ", site, "| origins:", len(v["Y"]))
        for name, p in (("persistence", v["persistence"]), ("repeat today", v["repeat"]), ("A (hybrid)", v["A_q"][:, 1, :])):
            r = E.mae_row(p, v["Y"], v["blocks"])
            print(f"    {name:13s}", "   ".join(f"+{h+1}d {m:5.2f} [{lo:5.2f},{hi:5.2f}]" for h, (m, lo, hi) in enumerate(r)))
        print("    coverage A:", np.round(E.coverage(v["A_q"], v["Y"]), 3))


##### (i) published codes


  Bejaia | origins: 116
    persistence   +1d  2.85 [ 2.02, 3.69]   +2d  3.97 [ 2.70, 5.38]   +3d  4.68 [ 3.40, 6.00]
    repeat today  +1d  2.99 [ 2.09, 3.89]   +2d  4.52 [ 3.14, 6.03]   +3d  5.47 [ 4.14, 6.80]
    A (hybrid)    +1d  3.05 [ 2.21, 3.86]   +2d  5.25 [ 4.08, 6.33]   +3d  6.75 [ 5.58, 8.00]
    coverage A: [0.543 0.517 0.457]
  SidiBelAbbes | origins: 116
    persistence   +1d  4.76 [ 3.81, 5.61]   +2d  7.22 [ 5.54, 8.63]   +3d  7.99 [ 6.08, 9.52]
    repeat today  +1d  5.38 [ 4.44, 6.25]   +2d  7.82 [ 6.45, 9.00]   +3d  9.24 [ 7.26,10.88]
    A (hybrid)    +1d  4.60 [ 3.81, 5.32]   +2d  5.93 [ 5.02, 6.87]   +3d  6.18 [ 5.28, 6.84]
    coverage A: [0.414 0.397 0.371]

##### (ii) own engine on real weather


  Bejaia | origins: 116
    persistence   +1d  3.49 [ 2.80, 4.34]   +2d  5.12 [ 4.13, 6.31]   +3d  6.07 [ 4.84, 7.71]
    repeat today  +1d  3.40 [ 2.61, 4.35]   +2d  4.99 [ 4.04, 6.25]   +3d  6.25 [ 4.89, 8.00]
    A (hybrid)    +1d  3.34 [ 2.72, 4.03]   +2d  5.17 [ 4.13, 6.46]   +3d  6.13 [ 4.38, 8.18]
    coverage A: [0.905 0.879 0.862]
  SidiBelAbbes | origins: 116
    persistence   +1d  6.51 [ 5.25, 7.80]   +2d  9.65 [ 7.64,12.02]   +3d 11.31 [ 9.04,14.49]
    repeat today  +1d  6.46 [ 5.17, 7.79]   +2d  9.80 [ 7.69,12.18]   +3d 11.54 [ 9.20,14.74]
    A (hybrid)    +1d  6.91 [ 5.75, 8.15]   +2d 10.15 [ 7.74,12.50]   +3d 11.21 [ 8.55,13.60]
    coverage A: [0.621 0.466 0.414]


## 4. Run-time cost

In [5]:
s = ss[45]; i = 110
f = FwiForecaster(clim, 100, seed=1)
args = ((s["ffmc"][i], s["dmc"][i], s["dc"][i]), {k: s[k][i] for k in ("temp", "rh", "wind", "rain")}, int(s["doy"][i]))
f.forecast(*args)
t0 = time.perf_counter()
for _ in range(20): f.forecast(*args)
print("100 members x 3 days: %.1f ms per forecast (cloud CPU, not VIM4)" % ((time.perf_counter() - t0) / 20 * 1e3))

100 members x 3 days: 1.8 ms per forecast (cloud CPU, not VIM4)
